In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')


/kaggle/input/datasets/rasaang/heartlist/heart.csv
/kaggle/input/datasets/rasaang/homework-1/Linear_Regression_Salary_Fill_in_the_Blanks.ipynb
/kaggle/input/datasets/rasaang/homework-1/Salary_Data.csv


In [2]:
# Rasaan Gouldbourne - AI 240
# Homework 2: Classification (Heart Disease Prediction)

import os
import warnings
import numpy as np
import pandas as pd

# Use non-interactive backend so plots save cleanly without needing a GUI/Jupyter
import matplotlib
matplotlib.use('Agg') 
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

warnings.filterwarnings('ignore')

# Set Kaggle standard directories
INPUT_DIR = '/kaggle/input'
OUTPUT_DIR = '/kaggle/working'
os.makedirs(OUTPUT_DIR, exist_ok=True)


def load_data(input_dir: str) -> pd.DataFrame:
    """Scans Kaggle input directory for CSV/TXT dataset or falls back to local path."""
    print("=" * 60)
    print("TASK 1: SCANNING AND LOADING DATASET")
    print("=" * 60)

    csv_path = None
    for root, _, files in os.walk(input_dir):
        for file in files:
            if file.endswith(('.csv', '.txt')):
                csv_path = os.path.join(root, file)
                break
        if csv_path:
            break

    if csv_path and os.path.exists(csv_path):
        print(f"Dataset successfully located at: {csv_path}")
        df = pd.read_csv(csv_path, sep=None, engine='python')
    else:
        raise FileNotFoundError(
            f"No dataset found in {input_dir}. Please attach the Heart Disease dataset to your Kaggle environment."
        )

    # Standardize column headers
    df.columns = [col.lower().strip() for col in df.columns]
    if 'target' not in df.columns:
        df.rename(columns={df.columns[-1]: 'target'}, inplace=True)

    print("\n--- Top 5 Rows of Patient Data Matrix ---")
    print(df.head())
    return df


def generate_exploration_charts(df: pd.DataFrame, output_dir: str) -> None:
    """Generates and saves exploratory data analysis charts to Kaggle output directory."""
    print("\n" + "=" * 60)
    print("TASK 2: SAVING EXPLORATION CHARTS")
    print("=" * 60)

    # Chart 1: Distribution of Target Variable
    plt.figure(figsize=(6, 4))
    sns.countplot(x='target', hue='target', data=df, palette='Set2', legend=False)
    plt.title('Distribution of Target Variable (Heart Disease)')
    plt.xlabel('0 = No Disease, 1 = Heart Disease')
    plt.ylabel('Patient Records Count')
    plt.grid(axis='y', linestyle=':', alpha=0.6)
    plt.tight_layout()
    target_plot_path = os.path.join(output_dir, 'target_distribution.png')
    plt.savefig(target_plot_path, dpi=300)
    plt.close()
    print(f"Saved: {target_plot_path}")

    # Chart 2: Feature Correlation Heatmap Matrix
    plt.figure(figsize=(12, 8))
    numeric_df = df.select_dtypes(include=[np.number])
    sns.heatmap(numeric_df.corr(), annot=True, cmap='coolwarm', fmt='.2f', linewidths=0.5)
    plt.title('Patient Feature Correlation Matrix Map')
    plt.tight_layout()
    corr_plot_path = os.path.join(output_dir, 'correlation_heatmap.png')
    plt.savefig(corr_plot_path, dpi=300)
    plt.close()
    print(f"Saved: {corr_plot_path}")


def preprocess_and_train(df: pd.DataFrame):
    """Cleans data, splits into train/test sets, scales features, and trains Logistic Regression."""
    print("\n" + "=" * 60)
    print("TASK 3: PREPROCESSING & TRAINING LOGISTIC REGRESSION")
    print("=" * 60)

    print("Checking for missing values across entries:")
    print(df.isnull().sum())

    df = df.dropna().copy()

    # Encode categorical columns if any non-numeric features exist
    X = pd.get_dummies(df.drop(columns=['target']), drop_first=True)
    y = df['target'].astype(int)

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42, stratify=y
    )

    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)

    model = LogisticRegression(random_state=42, max_iter=1000)
    model.fit(X_train_scaled, y_train)
    print("\nOptimization Phase Complete: Model weights converged successfully.")

    return model, X_test_scaled, y_test


def evaluate_model(model, X_test_scaled: np.ndarray, y_test: pd.Series, output_dir: str) -> None:
    """Evaluates model performance, saves metrics report and confusion matrix plot."""
    print("\n" + "=" * 60)
    print("TASK 4: MODEL EVALUATION METRICS REPORT")
    print("=" * 60)

    y_pred = model.predict(X_test_scaled)
    accuracy = accuracy_score(y_test, y_pred)
    report = classification_report(y_test, y_pred)
    cm = confusion_matrix(y_test, y_pred)

    print(f"Overall Testing Classification Accuracy: {accuracy:.4f} ({accuracy * 100:.2f}%)")
    print("\nClassification Metrics Breakdown Matrix:")
    print(report)
    print("Confusion Matrix Output Array:")
    print(cm)

    # Save text metrics report to Kaggle working directory
    report_path = os.path.join(output_dir, 'evaluation_report.txt')
    with open(report_path, 'w') as f:
        f.write(f"Overall Testing Classification Accuracy: {accuracy:.4f} ({accuracy * 100:.2f}%)\n\n")
        f.write("Classification Metrics Breakdown Matrix:\n")
        f.write(report + "\n")
        f.write("Confusion Matrix:\n")
        f.write(np.array2string(cm))
    print(f"\nSaved metrics report: {report_path}")

    # Save confusion matrix heatmap plot
    plt.figure(figsize=(5, 4))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=False)
    plt.title('Confusion Matrix Heatmap Breakdown')
    plt.xlabel('Predicted Diagnosis Label')
    plt.ylabel('True Diagnosis Label')
    plt.tight_layout()
    cm_plot_path = os.path.join(output_dir, 'confusion_matrix.png')
    plt.savefig(cm_plot_path, dpi=300)
    plt.close()
    print(f"Saved: {cm_plot_path}")
    print("=" * 60)


def main():
    df = load_data(INPUT_DIR)
    generate_exploration_charts(df, OUTPUT_DIR)
    model, X_test_scaled, y_test = preprocess_and_train(df)
    evaluate_model(model, X_test_scaled, y_test, OUTPUT_DIR)


if __name__ == "__main__":
    main()


TASK 1: SCANNING AND LOADING DATASET
Dataset successfully located at: /kaggle/input/datasets/rasaang/heartlist/heart.csv

--- Top 5 Rows of Patient Data Matrix ---
   age  sex  cp  trestbps  chol  fbs  restecg  thalach  exang  oldpeak  slope  \
0   52    1   0       125   212    0        1      168      0      1.0      2   
1   53    1   0       140   203    1        0      155      1      3.1      0   
2   70    1   0       145   174    0        1      125      1      2.6      0   
3   61    1   0       148   203    0        1      161      0      0.0      2   
4   62    0   0       138   294    1        1      106      0      1.9      1   

   ca  thal  target  
0   2     3       0  
1   0     3       0  
2   0     3       0  
3   1     3       0  
4   3     2       0  

TASK 2: SAVING EXPLORATION CHARTS
Saved: /kaggle/working/target_distribution.png
Saved: /kaggle/working/correlation_heatmap.png

TASK 3: PREPROCESSING & TRAINING LOGISTIC REGRESSION
Checking for missing values across